In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

/Users/emmanuel/Documents/belugas/beluga-call-pipeline


In [2]:
import pandas as pd

import torch
from models.resnet import ResnetMultilabel
from models.mobilenet import MobileNetMultilabel
from models.quant_mobilenet import load_mobilenet_v3_quant

from training.cross_validation import run_cross_val, train_model


## Running the Optimization Experiments

This section covers the model optimization experiments:
- Switching from **ResNet18** to **MobileNet V3 Small**,
- Further **Truncating** the MobileNet architecture,
- Applying **8-bit Quantization-Aware Training (QAT)** on the MobileNet model.

Each experiment is executed with **cross-validation** as described in the paper.  
Results are written to a dedicated `results/` directory and subsequently examined in the `results_analysis` folder.



In [3]:
labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")
labels_df["ClipFilenamePt"] = labels_df["clip_filename"].str.replace(".wav", ".pt", regex=False)


label_columns = ["ECHO", "HFPC", "BBPC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Verified_Dataset/spectrograms/"

results_dir = "./results/new_dataset"

In [4]:
training_config_default = {
    "batch_size": 32,
    "lr_decay_factor": 0.5,
    "patience_lr": 2,
    "n_epochs": 100, #100
    "min_epochs": 10, #15
    "patience_early_stopping": 5,
    "metric_mode": "max",
    "val_metric": "f1",
}

### Resnet18


In [26]:
run_cross_val(
    labels_df, 
    label_columns, 
    ResnetMultilabel,  
    processed_spects_dir,
    run_name="resnet",
    model_kwargs={
        "pretrained":True,
    }, 
    training_config=training_config_default,
    save_models=True,
    use_quantization=False,
)


Val Epoch: 30


100%|██████████| 59/59 [00:08<00:00,  6.92it/s]


Val Epoch: 30 Results - 
loss: 21.631, 
accuracy: {'Labels_Average': 0.9745694398880005, 'ECHO': 0.9677072167396545, 'HFPC': 0.9860064387321472, 'BBPC': 0.9817007780075073, 'Whistle': 0.9628632664680481}, 
f1: {'Labels_Average': 0.8858972191810608, 'ECHO': 0.9463327527046204, 'HFPC': 0.8725489974021912, 'BBPC': 0.8365384340286255, 'Whistle': 0.8881685733795166}, 
precision: {'Labels_Average': 0.8965355753898621, 'ECHO': 0.9566003680229187, 'HFPC': 0.8725489974021912, 'BBPC': 0.8787878751754761, 'Whistle': 0.8782051205635071}, 
recall: {'Labels_Average': 0.8763394951820374, 'ECHO': 0.9362831711769104, 'HFPC': 0.8725489974021912, 'BBPC': 0.7981651425361633, 'Whistle': 0.8983606696128845}, 
AUC: {'Labels_Average': 0.9912785887718201, 'ECHO': 0.9957921504974365, 'HFPC': 0.9918543100357056, 'BBPC': 0.9872902631759644, 'Whistle': 0.9901776313781738}, 
exact_match: {'Labels_Average': 0.9149623513221741},

Training Epoch: 31


100%|██████████| 233/233 [01:05<00:00,  3.58it/s]


Train Epoch: 31 Results - 
loss: 0.074, 
accuracy: {'Labels_Average': 0.9999663829803467, 'ECHO': 1.0, 'HFPC': 0.9998654127120972, 'BBPC': 1.0, 'Whistle': 1.0}, 
f1: {'Labels_Average': 0.999683141708374, 'ECHO': 1.0, 'HFPC': 0.9987325668334961, 'BBPC': 1.0, 'Whistle': 1.0}, 
precision: {'Labels_Average': 1.0, 'ECHO': 1.0, 'HFPC': 1.0, 'BBPC': 1.0, 'Whistle': 1.0}, 
recall: {'Labels_Average': 0.9993671178817749, 'ECHO': 1.0, 'HFPC': 0.9974683523178101, 'BBPC': 1.0, 'Whistle': 1.0}, 
AUC: {'Labels_Average': 0.9999998807907104, 'ECHO': 1.0, 'HFPC': 0.9999996423721313, 'BBPC': 1.0, 'Whistle': 1.0}, 
exact_match: {'Labels_Average': 0.9998654127120972},

Val Epoch: 31


...

100%|██████████| 12/12 [00:02<00:00,  4.68it/s]


Test on site cac Epoch: 0 Results - 
loss: 47.157, 
accuracy: {'Labels_Average': 0.9495798349380493, 'ECHO': 0.9719887971878052, 'HFPC': 0.9803921580314636, 'BBPC': 0.9495798349380493, 'Whistle': 0.8963585495948792}, 
f1: {'Labels_Average': 0.8810460567474365, 'ECHO': 0.9793388247489929, 'HFPC': 0.8727272748947144, 'BBPC': 0.7692307829856873, 'Whistle': 0.9028871655464172}, 
precision: {'Labels_Average': 0.9283804893493652, 'ECHO': 0.9673469662666321, 'HFPC': 0.8888888955116272, 'BBPC': 0.9375, 'Whistle': 0.9197860956192017}, 
recall: {'Labels_Average': 0.8468866348266602, 'ECHO': 0.991631805896759, 'HFPC': 0.8571428656578064, 'BBPC': 0.6521739363670349, 'Whistle': 0.8865979313850403}, 
AUC: {'Labels_Average': 0.9592701196670532, 'ECHO': 0.993227481842041, 'HFPC': 0.964068591594696, 'BBPC': 0.9156297445297241, 'Whistle': 0.9641546607017517}, 
exact_match: {'Labels_Average': 0.8151260614395142},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  \
0  CAC_2021

100%|██████████| 4/4 [00:01<00:00,  3.19it/s]


Test on site rdl Epoch: 0 Results - 
loss: 33.098, 
accuracy: {'Labels_Average': 0.9451754093170166, 'ECHO': 0.9122806787490845, 'HFPC': 0.9649122953414917, 'BBPC': 0.9298245906829834, 'Whistle': 0.9736841917037964}, 
f1: {'Labels_Average': 0.8839057683944702, 'ECHO': 0.800000011920929, 'HFPC': 0.9259259104728699, 'BBPC': 0.8399999737739563, 'Whistle': 0.9696969985961914}, 
precision: {'Labels_Average': 0.8919642567634583, 'ECHO': 0.800000011920929, 'HFPC': 0.8928571343421936, 'BBPC': 0.875, 'Whistle': 1.0}, 
recall: {'Labels_Average': 0.8776018023490906, 'ECHO': 0.800000011920929, 'HFPC': 0.9615384340286255, 'BBPC': 0.807692289352417, 'Whistle': 0.9411764740943909}, 
AUC: {'Labels_Average': 0.9823586940765381, 'ECHO': 0.9653932452201843, 'HFPC': 0.9860140085220337, 'BBPC': 0.9798950552940369, 'Whistle': 0.998132586479187}, 
exact_match: {'Labels_Average': 0.8333333134651184},
                   Filename Site  ECHO_true  ECHO_pred    ECHO_probs  \
0  RDL_20200722_10411693.pt  RDL      

,Filename,Site,ECHO_true,ECHO_pred,ECHO_probs,HFPC_true,HFPC_pred,HFPC_probs,BBPC_true,BBPC_pred,BBPC_probs,Whistle_true,Whistle_pred,Whistle_probs
0,RDL_20200722_10425555.pt,RDL,0.0,1.0,9.965374e-01,0.0,0.0,4.644563e-09,0.0,0.0,9.475898e-06,0.0,0.0,1.314030e-06
1,RDL_20200722_10454327.pt,RDL,0.0,0.0,1.969475e-11,0.0,0.0,1.188055e-15,0.0,0.0,1.260993e-11,0.0,0.0,1.237506e-09
2,RDL_20200722_10520000.pt,RDL,0.0,0.0,2.245798e-01,0.0,0.0,8.878233e-05,1.0,1.0,9.999946e-01,1.0,1.0,9.996926e-01
3,RDL_20200722_10525400.pt,RDL,0.0,0.0,1.175927e-06,0.0,0.0,2.011259e-08,1.0,1.0,1.000000e+00,1.0,1.0,9.999368e-01
4,RDL_20200722_10531800.pt,RDL,0.0,0.0,1.841447e-04,0.0,0.0,1.052133e-04,1.0,1.0,9.999887e-01,1.0,1.0,9.999274e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
565,RDL_20200901_19125986.pt,RDL,0.0,0.0,1.882238e-15,0.0,0.0,2.117531e-17,0.0,0.0,3.708723e-16,0.0,0.0,3.814236e-17
566,RDL_20200906_14553296.pt,RDL,0.0,0.0,1.059103e-15,0.0,0.0,1.530743e-17,0.0,0.0,1.470347e-14,0.0,0.0,2.229635e-15
567,RDL_20200906_14555838.pt,RDL,0.0,0.0,5.848732e-13,0.0,0.0,4.505641e-13,0.0,0.0,3.944376e-13,0.0,0.0,9.961523e-13
568,RDL_20200906_14555938.pt,RDL,0.0,0.0,4.840709e-14,0.0,0.0,1.333679e-15,0.0,0.0,3.634287e-15,0.0,0.0,1.684795e-16


#### Running all MobileNet variants: layer depth & quantization 

In [5]:

n_layers_to_test = [8, 6,  10, 12,]  
# n_layers_to_test = [2, 4, 6, 8, 10, 12,]  
quantization_options = [False]

for n_layers in n_layers_to_test:
    for use_quantization in quantization_options:
        # Create run name based on parameters
        quant_suffix = "_qat" if use_quantization else ""
        run_name = f"mobile_net{quant_suffix}_{n_layers}_layers"
        
        print(f"\n{'='*80}")
        print(f"Running experiment: {run_name}")
        print(f"n_layers: {n_layers}, quantization: {use_quantization}")
        print(f"{'='*80}")

        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model_kwargs = {
            "pretrained": True,
            "n_layers": n_layers
        }

        if use_quantization:
            model_kwargs["qat"] = True
        
        try:
            run_cross_val(
                labels_df, 
                label_columns, 
                model_class,  
                processed_spects_dir,
                run_name=run_name,
                model_kwargs=model_kwargs, 
                n_splits=5,
                training_config=training_config_default,
                save_models=True,
                use_quantization=use_quantization,
            )
            print(f"✅ Successfully completed: {run_name}")
            
        except Exception as e:
            print(f"❌ Error in experiment {run_name}: {str(e)}")
            print(f"Continuing with next experiment...")
            continue

print(f"\n{'='*80}")
print("All experiments completed!")
print(f"{'='*80}")


Val Epoch: 25


100%|██████████| 59/59 [00:07<00:00,  8.30it/s]


Val Epoch: 25 Results - 
loss: 14.762, 
accuracy: {'Labels_Average': 0.9759149551391602, 'ECHO': 0.9703983068466187, 'HFPC': 0.9881592988967896, 'BBPC': 0.9773950576782227, 'Whistle': 0.9677072167396545}, 
f1: {'Labels_Average': 0.8938934803009033, 'ECHO': 0.9513704776763916, 'HFPC': 0.8910890817642212, 'BBPC': 0.8292682766914368, 'Whistle': 0.9038461446762085}, 
precision: {'Labels_Average': 0.8834489583969116, 'ECHO': 0.9539006948471069, 'HFPC': 0.8571428656578064, 'BBPC': 0.8159999847412109, 'Whistle': 0.9067524075508118}, 
recall: {'Labels_Average': 0.905155599117279, 'ECHO': 0.948853611946106, 'HFPC': 0.9278350472450256, 'BBPC': 0.8429751992225647, 'Whistle': 0.9009584784507751}, 
AUC: {'Labels_Average': 0.9934185743331909, 'ECHO': 0.9952896237373352, 'HFPC': 0.9957790374755859, 'BBPC': 0.9914025068283081, 'Whistle': 0.9912031888961792}, 
exact_match: {'Labels_Average': 0.9176533818244934},
New best model found at epoch 25 with f1: 0.8939

Training Epoch: 26


100%|██████████| 233/233 [00:54<00:00,  4.30it/s]


Train Epoch: 26 Results - 
loss: 0.434, 
accuracy: {'Labels_Average': 0.9990577697753906, 'ECHO': 0.9994615912437439, 'HFPC': 0.9991923570632935, 'BBPC': 0.9983847141265869, 'Whistle': 0.9991923570632935}, 
f1: {'Labels_Average': 0.9941892623901367, 'ECHO': 0.9991386532783508, 'HFPC': 0.9929078221321106, 'BBPC': 0.9870689511299133, 'Whistle': 0.9976415038108826}, 
precision: {'Labels_Average': 0.9943686127662659, 'ECHO': 0.9987085461616516, 'HFPC': 0.9905660152435303, 'BBPC': 0.9913420081138611, 'Whistle': 0.9968578219413757}, 
recall: {'Labels_Average': 0.9940222501754761, 'ECHO': 0.9995691776275635, 'HFPC': 0.9952606558799744, 'BBPC': 0.9828326106071472, 'Whistle': 0.9984264373779297}, 
AUC: {'Labels_Average': 0.9999803304672241, 'ECHO': 0.9999988675117493, 'HFPC': 0.9999847412109375, 'BBPC': 0.9999673366546631, 'Whistle': 0.9999704957008362}, 
exact_match: {'Labels_Average': 0.996230959892273},

Val Epoch: 26


...

Train Epoch: 27 Results - 
loss: 0.389, 
accuracy: {'Labels_Average': 0.9992596507072449, 'ECHO': 0.9993269443511963, 'HFPC': 0.9997307658195496, 'BBPC': 0.9987885355949402, 'Whistle': 0.9991923570632935}, 
f1: {'Labels_Average': 0.9961313009262085, 'ECHO': 0.9989221692085266, 'HFPC': 0.9976303577423096, 'BBPC': 0.9903329610824585, 'Whistle': 0.9976396560668945}, 
precision: {'Labels_Average': 0.9965591430664062, 'ECHO': 0.9995685815811157, 'HFPC': 0.9976303577423096, 'BBPC': 0.9913978576660156, 'Whistle': 0.9976396560668945}, 
recall: {'Labels_Average': 0.9957042336463928, 'ECHO': 0.9982765913009644, 'HFPC': 0.9976303577423096, 'BBPC': 0.9892703890800476, 'Whistle': 0.9976396560668945}, 
AUC: {'Labels_Average': 0.9999897480010986, 'ECHO': 0.9999954104423523, 'HFPC': 0.9999969005584717, 'BBPC': 0.9999759197235107, 'Whistle': 0.9999908208847046}, 
exact_match: {'Labels_Average': 0.9970386028289795},

Val Epoch: 27


100%|██████████| 59/59 [00:07<00:00,  8.23it/s]


Val Epoch: 27 Results - 
loss: 15.306, 
accuracy: {'Labels_Average': 0.9755113124847412, 'ECHO': 0.9687836170196533, 'HFPC': 0.9881592988967896, 'BBPC': 0.9773950576782227, 'Whistle': 0.9677072167396545}, 
f1: {'Labels_Average': 0.8920858502388, 'ECHO': 0.9485815763473511, 'HFPC': 0.8899999856948853, 'BBPC': 0.824999988079071, 'Whistle': 0.9047619104385376}, 
precision: {'Labels_Average': 0.8871796131134033, 'ECHO': 0.9536541700363159, 'HFPC': 0.8640776872634888, 'BBPC': 0.831932783126831, 'Whistle': 0.8990536332130432}, 
recall: {'Labels_Average': 0.8974533677101135, 'ECHO': 0.9435626268386841, 'HFPC': 0.9175257682800293, 'BBPC': 0.8181818127632141, 'Whistle': 0.9105431437492371}, 
AUC: {'Labels_Average': 0.9929580092430115, 'ECHO': 0.9949514865875244, 'HFPC': 0.9953166246414185, 'BBPC': 0.9910266399383545, 'Whistle': 0.9905372858047485}, 
exact_match: {'Labels_Average': 0.9149623513221741},

Training Epoch: 28


 50%|████▉     | 116/233 [00:27<00:28,  4.16it/s]


KeyboardInterrupt: 

<h4>Function call template to run quick experiments<h4>


In [ ]:
run_cross_val(
    labels_df, 
    label_columns, 
    MobileNetMultilabel,  
    processed_spects_dir,
    run_name="mobile_net_hp_1024_8_layers_all_absences",
    model_kwargs={
        "pretrained":True,
        "n_layers": 8
    }, 
    n_splits=5,
    training_config=training_config_default,
    save_models=True,
    use_quantization=False,
)

## Site Generalization Experiments


1. **Site-specific models** — train a separate model per site.
2. **Leave-One-Site-Out** — train on all but one site and test on the held-out site to assess generalizability.

All runs follow the protocol described in the paper.


In [9]:
from training.cross_validation import create_test_fold_indices
from sklearn.model_selection import KFold, train_test_split
from models.utils import aggregate_folds_testing_metrics



labels_df = pd.read_csv("../data/labels/Overlaps_1s.csv")
labels_df["ClipFilenamePt"] = labels_df["ClipFilename"] + ".pt"


label_columns = ["ECHO", "HFPC", "CC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Full_Dataset/Overlaps_1s_hp_1024_resize/"


labels_df = create_test_fold_indices(labels_df, 5)

### Site-specific models

In [ ]:
all_sites = ["RDL", "CAC", "BSM", "KAM" ]

use_quantization = False

for train_site in all_sites:

    for fold_idx in range(5):
        
        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model = model_class(
            pretrained=True,
            n_layers=8,
            num_classes=len(label_columns)
        )

        train_site_df = labels_df[labels_df["Site"]==train_site]
        train_data = train_site_df[train_site_df["test_fold_idx"] != fold_idx]
        train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['Site'])
        
        test_data = train_site_df[train_site_df["test_fold_idx"] == fold_idx]

        run_name = f"{train_site}_only"
        if use_quantization:
            run_name = run_name + "_qat"


        run_dir, _, _ = train_model(
            labels_df,
            label_columns,
            model,
            train_data,
            val_data,
            test_data,
            processed_spects_dir=processed_spects_dir,
            fold_idx=fold_idx,
            results_dir="./results/sites_generalization",
            run_name=run_name,
            training_config=training_config_default,
            use_quantization=use_quantization,
            compute_sites_metrics=True
        )

    aggregate_folds_testing_metrics(run_dir)

### Leave One Site out

In [ ]:


all_sites = ["BSM", "RDL", "CAC", "KAM" ]

use_quantization = False

for out_site in all_sites:
    for fold_idx in range(5):
        
        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model = model_class(
            pretrained=True,
            n_layers=8,
            num_classes=len(label_columns)
        )

        train_sites = [site for site in all_sites if site != out_site]

        train_df = labels_df[labels_df["test_fold_idx"] != fold_idx]
        
        train_sites_df = train_df[train_df["Site"].isin(train_sites)]

        train_data, val_data = train_test_split(train_sites_df, test_size=0.2, random_state=42, stratify=train_sites_df['Site'])
        
        test_data = labels_df[labels_df["test_fold_idx"] == fold_idx]


        run_name = f"leave_{out_site}_out"
        if use_quantization:
            run_name = run_name + "_qat"

        print(run_name)
        print(f"Site out : {out_site}")
        print("Train df")
        print(train_data["Site"].value_counts())
        print("\nVal df")
        print(val_data["Site"].value_counts())

        # break
        run_dir, _, _ = train_model(
            labels_df,
            label_columns,
            model,
            train_data,
            val_data,
            test_data,
            processed_spects_dir=processed_spects_dir,
            fold_idx=fold_idx,
            run_name=run_name,
            results_dir="./results/sites_generalization",
            training_config=training_config_default,
            use_quantization=use_quantization,
            
        )
    
    aggregate_folds_testing_metrics(run_dir)

    # break

<h2>Training the final model on all the data</h2>

In [ ]:
from training.cross_validation import create_test_fold_indices
from sklearn.model_selection import KFold, train_test_split
from models.utils import aggregate_folds_testing_metrics



labels_df = pd.read_csv("../data/labels/Overlaps_1s.csv")
labels_df["ClipFilenamePt"] = labels_df["ClipFilename"] + ".pt"


label_columns = ["ECHO", "HFPC", "CC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Full_Dataset/Overlaps_1s_hp_1024_resize/"

results_dir = "./final_results"

labels_df = create_test_fold_indices(labels_df, 5)

In [ ]:

use_quantization = True
        
model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

model = model_class(
    pretrained=True,
    n_layers=8,
    num_classes=len(label_columns)
)

train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['Site'])
test_data = val_data #Doesn't matter here, won't be used anyway

run_name = f"Final_model"
if use_quantization:
    run_name = run_name + "_qat"

run_dir = train_model(
    labels_df,
    label_columns,
    model,
    train_data,
    val_data,
    test_data,
    fold_idx=0,
    processed_spects_dir=processed_spects_dir,
    run_name=run_name,
    results_dir="results/final_model",
    training_config=training_config_default,
    use_quantization=use_quantization,
    save_model=True
)